# Using Own Data

The method 

This notebook:
1. Writes an example data directory, using the SIR simulator as a stand-in
   for an external simulator, generates some example training and contaminated data. 
2. Trains a surrogate density model `EnergyBasedModel`.
5. Calibrates $\beta$ with GPC and computes the calibrated posterior.


In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import torch

from nsm_bayes.inference.conjugate.calibrate_beta import calibrate_beta_gpc
from nsm_bayes.models.conjugate import EnergyBasedModel
from nsm_bayes.models.conjugate.conj_posterior import ConjugatePosterior
from nsm_bayes.simulators.benchmark_simulators.sir import SIRSimulator
from nsm_bayes.utils import set_seed

# Repository root, whether the notebook is run from examples/ or the root
REPO = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent

set_seed(123)

## 1. Generating Test Data

In order to perform inference, we need the following data. 

For training the network: 
- `theta.pkl` of shape `(N, d_theta)`. These are samples from the prior, used for training a surrogate model. 
- `x_sim.pkl` of shape `(N, d_x)`. These are simulations corresponding to the prior samples, used for training the surrogate density model. 

For inference  
- `x_obs.pkl` of shape `(M, d_x)`. This is the observed data.
- `prior_mean` and `prior_cov` with shapes `(d_theta,)` and `(d_theta, d_theta)` respectively. In the conjugate method we assume a Gaussian prior. These can be also computed from the `theta.pkl` data. 

In practice these can come from any simulator. Here we create placeholder demonstration data with the SIR simulator, exactly as the SIR experiment does: 50,000 prior draws with their summary statistics, and 1,000 clean observations at the true parameter. 

In [ ]:
#generating some dummy data files
#im using the sir defaults, but they could be anything. 
data_dir = REPO / "data" / "sir_pickle"
data_dir.mkdir(parents=True, exist_ok=True)

def to_unconstrained(values):
    """(beta, gamma, rho, I0) -> (log beta, log gamma, logit rho, log I0)."""
    v = torch.tensor(values, dtype=torch.float32)
    return torch.stack([v[0].log(), v[1].log(), torch.logit(v[2]), v[3].log()])

# The prior mean and true parameter are on the natural scale, so both are
# transformed; the prior standard deviations are already in unconstrained space.
sir = SIRSimulator(
    prior_mean=to_unconstrained([0.5, 0.2, 0.5, 20.0]),
    prior_cov=torch.diag(torch.tensor([0.5, 0.5, 1.0, 0.7]) ** 2),
)
theta_true = to_unconstrained([0.6, 0.15, 0.6, 20.0])

theta, y_sim = sir.generate_training_data(1000)

#generate misspecified observation data 
x_obs_mis,_ = sir.generate_contaminated_data(theta_true, n_obs=100, epsilon=0.05)

arrays = {
    "theta": theta,
    "x_sim": sir.summary(y_sim),
    "x_obs": sir.summary(x_obs_mis),
    "theta_true": theta_true,
    "prior_mean": sir.prior_mean,
    "prior_cov": sir.prior_cov,
}

#save them 
for name, arr in arrays.items():
    with open(data_dir / f"{name}.pkl", "wb") as fh:
        pickle.dump(arr, fh)
    print(f"{name}.pkl: {tuple(arr.shape)}")

## 2. Training the conjugate model

`EnergyBasedModel` fits standardizers on the simulations and trains the
networks $T_\phi$ and $b_\phi$ by score matching, giving the surrogate
likelihood

\begin{equation*}
q_\phi(x \mid \theta) \propto \exp\{T_\phi(x)^\top \theta + b_\phi(x)\}.
\end{equation*}

In [ ]:
#load the training data
from nsm_bayes.simulators.fromdata import load_pkl

#define the training data from the files
theta = load_pkl(str(data_dir)+"/theta.pkl")
x_sim = load_pkl(str(data_dir)+"/x_sim.pkl")

#shortcuts for the dimensions
d_x = x_sim.shape[1]
d_theta = theta.shape[1]

#set up the model
model = EnergyBasedModel(d_x=d_x, d_theta=d_theta, hidden_dim=16)

#pass the training data and fit the surrogate density 
#save the checkpoint in save dir. 
model.fit(x_sim=x_sim, theta_sim=theta,save_file=str(data_dir)+"/sir_model.pt")

model.load_checkpoint()


In [ ]:

history = model.train_history
plt.figure(figsize=(7, 3.5))
plt.plot(history["train_losses"], label="train")
plt.plot(history["val_losses"], label="validation")
plt.xlabel("epoch")
plt.ylabel("score-matching loss")
plt.legend()
plt.grid(True)
plt.show()

## 5. Calibrating $\beta$

The score-matching loss is not a log-likelihood, so the generalised
posterior $\pi(\theta)\,\exp\{-\beta\, n\, L(\theta; x)\}$ needs a
temperature $\beta$. Too large a $\beta$ gives an overconfident posterior;
too small a $\beta$ returns the prior. `calibrate_beta_gpc` tunes $\beta$
by general posterior calibration (GPC), matching the bootstrap coverage
of the $1 - \alpha$ credible region to its nominal level.

`robust=True` in `ConjugatePosterior` switches to robust (IMQ-weighted)
estimates of the data statistics.

In [ ]:
from nsm_bayes.simulators.fromdata import gaussian_mle_from_data

In [ ]:
#load saved values
prior_mean = load_pkl(str(data_dir)+"/prior_mean.pkl")
prior_cov = load_pkl(str(data_dir)+"/prior_cov.pkl")

#or estimate from the data
prior_mean, prior_cov = gaussian_mle_from_data(theta)

posterior = ConjugatePosterior(
    model,
    prior_mean,
    prior_cov,
    robust=True,
    weight_type="imq",
)

beta, gpc_history = calibrate_beta_gpc(
    x_obs_mis,
    posterior,
    initial_beta=0.01,
    target_coverage=0.95,
    beta_min=1e-4,
)
print(f"calibrated beta: {beta:.4g}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(gpc_history["betas"], "o-")
ax1.set_yscale("log")
ax1.set_xlabel("GPC iteration")
ax1.set_ylabel(r"$\beta$")
ax2.plot(gpc_history["coverages"], "o-", label="bootstrap coverage")
ax2.axhline(0.95, color="k", ls="--", label="target 0.95")
ax2.set_xlabel("GPC iteration")
ax2.set_ylabel("coverage")
ax2.set_ylim(0, 1.05)
ax2.legend()
plt.tight_layout()
plt.show()

## 6. Calibrated posterior

We check the posterior wiht the calibrated $\beta$

In [ ]:
mu_n, Sigma_n = posterior.calibrated_posterior(x_obs_mis, beta)
mu_n = mu_n.detach()
std_n = Sigma_n.detach().diagonal().sqrt()

labels = [r"$\log \beta$", r"$\log \gamma$", r"$\mathrm{logit}\,\rho$", r"$\log I_0$"]
for lab, m, s, t in zip(labels, mu_n, std_n, theta_true):
    print(f"{lab:>22}: {m:7.3f} ± {s:.3f}   (true {t:7.3f})")

fig, axes = plt.subplots(1, d_theta, figsize=(13, 3))
for i, ax in enumerate(axes):
    grid = torch.linspace(min(mu_n[i] - 4 * std_n[i], theta_true[i]),
                          max(mu_n[i] + 4 * std_n[i], theta_true[i]), 300)
    pdf = torch.distributions.Normal(mu_n[i], std_n[i]).log_prob(grid).exp()
    prior_pdf = torch.distributions.Normal(prior.loc[i], prior.covariance_matrix[i, i].sqrt()).log_prob(grid).exp()
    ax.plot(grid, pdf, label="posterior")
    ax.plot(grid, prior_pdf, color="0.6", ls=":", label="prior")
    ax.axvline(theta_true[i], color="r", ls="--", label="true")
    ax.set_title(labels[i])
    ax.set_yticks([])
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()